In [1]:
#
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "amex-46887"
DATASET_ID = "amex_risk"

client = bigquery.Client(project=PROJECT_ID)

In [2]:
query = f"""
SELECT table_name
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name
0,sample_submission_raw
1,test_data_raw
2,train_data_raw
3,train_joined
4,train_labels_raw


In [3]:
# Check the number of rows in the train_data_raw table
query = f"""
SELECT COUNT(*) AS n_rows
FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
"""
client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_rows
0,5531451


In [4]:
# Unique Customers
query = f"""
SELECT COUNT(DISTINCT customer_ID) AS unique_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
"""
client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,unique_customers
0,458913


In [5]:
# Label Balance
query = f"""
SELECT target, COUNT(*) AS n_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_labels_raw`
GROUP BY target
ORDER BY target
"""
client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,target,n_customers
0,0,340085
1,1,118828


In [6]:
# Sequence Lengths
query = f"""
SELECT
  COUNT(*) AS n_statements,
  COUNT(DISTINCT customer_ID) AS n_customers
FROM `{PROJECT_ID}.{DATASET_ID}.train_data_raw`
GROUP BY customer_ID
LIMIT 20
"""
client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_statements,n_customers
0,13,1
1,13,1
2,13,1
3,1,1
4,13,1
5,13,1
6,13,1
7,13,1
8,13,1
9,13,1


In [7]:
query = open("../sql/02_join_labels.sql").read()
client.query(query).result()
print("train_joined created")

train_joined created


In [9]:
query = """
SELECT table_name
FROM `amex-46887.amex_risk.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

tables_df = client.query(query).to_dataframe()
tables_df

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name
0,sample_submission_raw
1,test_data_raw
2,train_data_raw
3,train_joined
4,train_labels_raw


In [10]:
query = """
CREATE OR REPLACE TABLE `amex-46887.amex_risk.train_joined` AS
SELECT
  t.*,
  l.target
FROM `amex-46887.amex_risk.train_data_raw` t
JOIN `amex-46887.amex_risk.train_labels_raw` l
USING (customer_ID)
"""

client.query(query).result()
print("train_joined created successfully")

train_joined created successfully


In [11]:
query = """
CREATE OR REPLACE TABLE `amex-46887.amex_risk.customer_features_base` AS
SELECT
  customer_ID,
  ANY_VALUE(target) AS target,
  COUNT(*) AS n_statements,

  AVG(P_2) AS P_2_avg,
  MIN(P_2) AS P_2_min,
  MAX(P_2) AS P_2_max,
  STDDEV(P_2) AS P_2_std,

  AVG(D_39) AS D_39_avg,
  MAX(D_39) AS D_39_max,
  STDDEV(D_39) AS D_39_std

FROM `amex-46887.amex_risk.train_joined`
GROUP BY customer_ID
"""

client.query(query).result()
print("customer_features_base created successfully")

customer_features_base created successfully


In [12]:
query = """
SELECT *
FROM `amex-46887.amex_risk.customer_features_base`
LIMIT 5
"""

df = client.query(query).to_dataframe()
df.head()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_ID,target,n_statements,P_2_avg,P_2_min,P_2_max,P_2_std,D_39_avg,D_39_max,D_39_std
0,e9c80296df4485a1a6ff842eb0ac639ac876fa5957d2ec...,0,1,0.757620,0.757620,0.757620,NaN,0.589621,0.589621,NaN
1,99f6625e9884c0c1e6f7754c92d6ea89e43e94f5f1adf2...,0,1,NaN,NaN,NaN,NaN,0.001540,0.001540,NaN
2,209aae461c0caef6a5f7df8e05d710ca9e7bcffd120eef...,0,1,0.629675,0.629675,0.629675,NaN,0.039126,0.039126,NaN
3,da875805ac37e0746b0dab48e29d18a385bef9c0e10dc4...,0,1,0.745530,0.745530,0.745530,NaN,0.005831,0.005831,NaN
4,8104aa170ce7945b6a24d4b0676a5a0e59c66729e122a0...,0,1,0.483792,0.483792,0.483792,NaN,0.005787,0.005787,NaN


In [13]:
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customer_ID) AS unique_customers
FROM `amex-46887.amex_risk.train_data_raw`
"""

client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_rows,unique_customers
0,5531451,458913


In [15]:
# Total rows and unique customers in labels
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT customer_ID) AS unique_customers
FROM `amex-46887.amex_risk.train_labels_raw`
"""

client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_rows,unique_customers
0,458913,458913


In [16]:
# Whether there any label customer_IDs that are missing in train_data_raw
query = """
SELECT COUNT(*) AS missing_customers
FROM `amex-46887.amex_risk.train_labels_raw` l
LEFT JOIN `amex-46887.amex_risk.train_data_raw` t
USING (customer_ID)
WHERE t.customer_ID IS NULL
"""

client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,missing_customers
0,0


In [17]:
# Distribution of number of statements per customer
query = """
WITH statement_counts AS (
  SELECT
    customer_ID,
    COUNT(*) AS n_statements
  FROM `amex-46887.amex_risk.train_data_raw`
  GROUP BY customer_ID
)
SELECT
  n_statements,
  COUNT(*) AS n_customers
FROM statement_counts
GROUP BY n_statements
ORDER BY n_statements
"""

statement_dist = client.query(query).to_dataframe()
statement_dist

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_statements,n_customers
0,1,5120
1,2,6098
2,3,5778
3,4,4673
4,5,4671
5,6,5515
6,7,5198
7,8,6110
8,9,6411
9,10,6721


In [18]:
# Basic Summary of statement counts
query = """
WITH statement_counts AS (
  SELECT
    customer_ID,
    COUNT(*) AS n_statements
  FROM `amex-46887.amex_risk.train_data_raw`
  GROUP BY customer_ID
)
SELECT
  MIN(n_statements) AS min_statements,
  MAX(n_statements) AS max_statements,
  AVG(n_statements) AS avg_statements
FROM statement_counts
"""

client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,min_statements,max_statements,avg_statements
0,1,13,12.053376


In [19]:
query = """
WITH statement_counts AS (
  SELECT
    customer_ID,
    COUNT(*) AS n_statements
  FROM `amex-46887.amex_risk.train_data_raw`
  GROUP BY customer_ID
)
SELECT
  n_statements,
  COUNT(*) AS n_customers
FROM statement_counts
GROUP BY n_statements
ORDER BY n_statements
"""

statement_dist = client.query(query).to_dataframe()
statement_dist

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,n_statements,n_customers
0,1,5120
1,2,6098
2,3,5778
3,4,4673
4,5,4671
5,6,5515
6,7,5198
7,8,6110
8,9,6411
9,10,6721


In [20]:
query = """
WITH statement_counts AS (
  SELECT
    customer_ID,
    COUNT(*) AS n_statements
  FROM `amex-46887.amex_risk.train_data_raw`
  GROUP BY customer_ID
)
SELECT
  MIN(n_statements) AS min_statements,
  MAX(n_statements) AS max_statements,
  AVG(n_statements) AS avg_statements
FROM statement_counts
"""

client.query(query).to_dataframe()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,min_statements,max_statements,avg_statements
0,1,13,12.053376


In [21]:
query = """
CREATE OR REPLACE TABLE `amex-46887.amex_risk.customer_features_v2` AS
WITH base AS (
  SELECT
    customer_ID,
    target,
    S_2,
    P_2,
    D_39
  FROM `amex-46887.amex_risk.train_joined`
),
ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY customer_ID ORDER BY S_2 ASC) AS rn_asc,
    ROW_NUMBER() OVER (PARTITION BY customer_ID ORDER BY S_2 DESC) AS rn_desc
  FROM base
),
agg AS (
  SELECT
    customer_ID,
    ANY_VALUE(target) AS target,
    COUNT(*) AS n_statements,

    AVG(P_2) AS P_2_avg,
    MIN(P_2) AS P_2_min,
    MAX(P_2) AS P_2_max,
    STDDEV(P_2) AS P_2_std,
    COUNT(P_2) AS P_2_non_null_count,
    COUNTIF(P_2 IS NULL) AS P_2_null_count,

    AVG(D_39) AS D_39_avg,
    MIN(D_39) AS D_39_min,
    MAX(D_39) AS D_39_max,
    STDDEV(D_39) AS D_39_std,
    COUNT(D_39) AS D_39_non_null_count,
    COUNTIF(D_39 IS NULL) AS D_39_null_count
  FROM base
  GROUP BY customer_ID
),
first_last AS (
  SELECT
    customer_ID,
    MAX(CASE WHEN rn_asc = 1 THEN P_2 END) AS P_2_first,
    MAX(CASE WHEN rn_desc = 1 THEN P_2 END) AS P_2_last,
    MAX(CASE WHEN rn_asc = 1 THEN D_39 END) AS D_39_first,
    MAX(CASE WHEN rn_desc = 1 THEN D_39 END) AS D_39_last
  FROM ranked
  GROUP BY customer_ID
)
SELECT
  a.*,
  f.P_2_first,
  f.P_2_last,
  f.P_2_last - f.P_2_first AS P_2_delta,
  f.D_39_first,
  f.D_39_last,
  f.D_39_last - f.D_39_first AS D_39_delta
FROM agg a
LEFT JOIN first_last f
USING (customer_ID)
"""

client.query(query).result()
print("customer_features_v2 created successfully")

customer_features_v2 created successfully


In [22]:
query = """
SELECT *
FROM `amex-46887.amex_risk.customer_features_v2`
LIMIT 5
"""

df_v2 = client.query(query).to_dataframe()
df_v2.head()

/home/dd4real2k/.pyenv/versions/alaid/lib/python3.14/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_ID,target,n_statements,P_2_avg,P_2_min,P_2_max,P_2_std,P_2_non_null_count,P_2_null_count,D_39_avg,...,D_39_max,D_39_std,D_39_non_null_count,D_39_null_count,P_2_first,P_2_last,P_2_delta,D_39_first,D_39_last,D_39_delta
0,e0f28803d5a24c95e5c7739744ba4463621a88714990a8...,0,13,NaN,NaN,NaN,NaN,0,13,0.004389,...,0.009282,0.002993,13,0,NaN,NaN,NaN,0.005627,0.000557,-0.005070
1,65f8678f6760f63aa07c1d3a8f3b0465846da136c7ea44...,0,1,NaN,NaN,NaN,NaN,0,1,0.000181,...,0.000181,NaN,1,0,NaN,NaN,NaN,0.000181,0.000181,0.000000
2,a458c50e661edcfe2df425ad56b5b0246d96e09cfb7862...,0,13,NaN,NaN,NaN,NaN,0,13,0.003591,...,0.009638,0.003451,13,0,NaN,NaN,NaN,0.000044,0.002632,0.002588
3,96b16e033ca6f75db4a49b89a65e3a846706110fbcff9b...,0,1,NaN,NaN,NaN,NaN,0,1,0.006405,...,0.006405,NaN,1,0,NaN,NaN,NaN,0.006405,0.006405,0.000000
4,20bbbfd1f2b49997ad9df0108a49fbc3e1ac71c2efb182...,0,4,NaN,NaN,NaN,NaN,0,4,0.005111,...,0.007492,0.002094,4,0,NaN,NaN,NaN,0.002472,0.005736,0.003264
